# 01 — Coleta e ingestão dos arquivos originais

Objetivo: preservar os três CSVs da Olist em um volume do Unity Catalog e registrar a carga. Entrada: arquivos baixados da fonte original. Saída: arquivos brutos intactos no volume. Pré-requisito: Databricks Free Edition com permissão no catálogo do workspace. Evidências: tela do upload, listagem do volume e inspeção inicial.

Execute a configuração e a criação do volume; faça o upload pela interface; então execute o restante. O comando Executar tudo só funciona depois que os três arquivos estão presentes.


## Fonte e licença

Fonte: Olist, Brazilian E-Commerce Public Dataset by Olist, Kaggle, https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce . Licença exibida na página da fonte: CC BY-NC-SA 4.0, https://creativecommons.org/licenses/by-nc-sa/4.0/ . Uso acadêmico com atribuição; os CSVs não entram no Git público. A página descreve pedidos de 2016 a 2018; as datas efetivas serão medidas no notebook 02.

Os arquivos pedidos contêm status e datas de compra, previsão e entrega; clientes contêm localidade; itens contêm frete por item. Isso sustenta as três perguntas condicionadas à qualidade e à cobertura das junções.


## Configuração do catálogo e do volume


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
for esquema in ["olist_mvp_raw", "olist_mvp_silver", "olist_mvp_gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.olist_mvp_raw.originais")
print(f"Volume pronto: {origem}")


## Upload pela interface

Baixe o conjunto original no Kaggle, extraia os três CSVs selecionados e, no Databricks, abra Novo → Adicionar ou carregar dados → Carregar arquivos para volume. Selecione o volume mostrado acima e envie os três arquivos sem editá-los. Registre no README a data real da coleta e inclua uma captura da tela de upload e outra dos arquivos no Explorador de Catálogo. Não use download pela rede do notebook: a Free Edition restringe internet de saída.


In [ ]:
inventario = {item.name: item.size for item in dbutils.fs.ls(origem)}
faltantes = sorted(set(arquivos.values()) - set(inventario))
assert not faltantes, f"Arquivos ausentes no volume: {faltantes}"
display(spark.createDataFrame(
    [(nome, inventario[nome]) for nome in arquivos.values()],
    ["arquivo", "tamanho_bytes"]
))


In [ ]:
import hashlib

hashes_esperados = {
    "olist_orders_dataset.csv": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf",
    "olist_order_items_dataset.csv": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279",
    "olist_customers_dataset.csv": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2",
}
def sha256_arquivo(caminho):
    digestao = hashlib.sha256()
    with open(caminho, "rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            digestao.update(bloco)
    return digestao.hexdigest()

for nome, esperado in hashes_esperados.items():
    observado = sha256_arquivo(f"{origem}/{nome}")
    assert observado == esperado, f"Arquivo divergente: {nome}"
    print(f"{nome}: SHA-256 conferido ({observado})")


## Leitura sem conversão de tipos


In [ ]:
def ler_origem(nome):
    caminho = f"{origem}/{arquivos[nome]}"
    quadro = (spark.read.option("header", "true")
              .option("inferSchema", "false")
              .option("sep", ",")
              .option("encoding", "UTF-8")
              .option("mode", "FAILFAST")
              .csv(caminho))
    assert quadro.columns == colunas_esperadas[nome], (
        f"Esquema inesperado em {arquivos[nome]}: {quadro.columns}"
    )
    return quadro


In [ ]:
for nome in arquivos:
    quadro = ler_origem(nome)
    print(f"{nome}: {quadro.count()} linhas; colunas: {quadro.columns}")
    quadro.printSchema()
    display(quadro.limit(5))
